# Mapa Aberto RJ — insights contraintuitivos auditáveis

**Pergunta.** Que fatos do período ago/2025–jul/2026 desafiam leituras simplistas sobre segurança no Rio?

**Fontes.** Série mensal por CISP do ISP-RJ; população e rendimento do Censo 2022 cruzados com as geometrias oficiais das CISPs. A análise mede registros, casos ou vítimas conforme o indicador — não risco individual nem pessoas únicas.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "public/data/crime-rio-snapshot.json").exists(), "Execute a partir da raiz do projeto"

def read_json(path):
    return json.loads((ROOT / path).read_text())

snapshot = read_json("public/data/crime-rio-snapshot.json")
population_raw = read_json("public/data/cisp-population.json")
territories_raw = read_json("public/data/cisp-neighborhoods.json")
income_raw = read_json("research/data/cisp-income-context.json")

rows = pd.DataFrame([
    {"cisp": row["cisp"], "period": row["period"], "phase": row["phase"], **row["values"]}
    for row in snapshot["rows"]
])
population = pd.DataFrame(population_raw["records"]).set_index("cisp")
territories = pd.DataFrame(territories_raw["records"]).set_index("cisp")
income = pd.DataFrame(income_raw["records"]).set_index("cisp")

CURRENT_START, CURRENT_END = "2025-08", "2026-07"
PREVIOUS_START, PREVIOUS_END = "2024-08", "2025-07"
current = rows.query('@CURRENT_START <= period <= @CURRENT_END').groupby('cisp').sum(numeric_only=True)
previous = rows.query('@PREVIOUS_START <= period <= @PREVIOUS_END').groupby('cisp').sum(numeric_only=True)

assert len(rows) == 1476 and current.index.nunique() == 41
assert current['letalidade_violenta'].sum() == 1589
assert population.population.sum() == 6211223
print({"rows": len(rows), "cisps": len(current), "latest": snapshot["latestPeriod"], "phase": snapshot["latestPhase"]})

{'rows': 1476, 'cisps': 41, 'latest': '2026-07', 'phase': [2]}


## 1. A cidade pode ter menos roubos e mais violência letal ao mesmo tempo

Comparar indicadores separados evita fabricar um “índice geral de segurança” que esconderia dinâmicas opostas.

In [2]:
indicators = ['total_roubos','roubo_rua','hom_doloso','letalidade_violenta','hom_por_interv_policial']
labels = {
    'total_roubos':'Total de roubos', 'roubo_rua':'Roubo de rua',
    'hom_doloso':'Homicídio doloso', 'letalidade_violenta':'Letalidade violenta',
    'hom_por_interv_policial':'Mortes por intervenção policial'
}
city = pd.DataFrame({
    'anterior': previous[indicators].sum(),
    'atual': current[indicators].sum(),
})
city['variacao_pct'] = (city.atual / city.anterior - 1) * 100
city.index = [labels[i] for i in city.index]
city.round(1)

,anterior,atual,variacao_pct
Total de roubos,66288,58944,-11.1
Roubo de rua,40093,33686,-16.0
Homicídio doloso,936,1016,8.5
Letalidade violenta,1372,1589,15.8
Mortes por intervenção policial,322,461,43.2


## 2. Uma CISP explica 47% do aumento municipal da letalidade

A CISP 22 cobre Penha, Olaria e partes de Brás de Pina e Penha Circular. A associação com a Operação Contenção é uma **inferência temporal e territorial**, não uma identificação individual de registros.

In [3]:
cisp = 22
city_delta = current.letalidade_violenta.sum() - previous.letalidade_violenta.sum()
cisp_delta = current.loc[cisp, 'letalidade_violenta'] - previous.loc[cisp, 'letalidade_violenta']
october = rows.query("cisp == 22 and period == '2025-10'").iloc[0]
result = {
    'letalidade_CISP22_anterior': int(previous.loc[cisp, 'letalidade_violenta']),
    'letalidade_CISP22_atual': int(current.loc[cisp, 'letalidade_violenta']),
    'participacao_no_aumento_municipal_pct': round(cisp_delta / city_delta * 100, 1),
    'mortes_intervencao_CISP22_out_2025': int(october.hom_por_interv_policial),
    'mortes_intervencao_CISP22_periodo': int(current.loc[cisp, 'hom_por_interv_policial']),
    'letalidade_Rio_sem_CISP22_variacao_pct': round(((current.letalidade_violenta.sum()-current.loc[cisp,'letalidade_violenta'])/(previous.letalidade_violenta.sum()-previous.loc[cisp,'letalidade_violenta'])-1)*100, 1),
}
result

{'letalidade_CISP22_anterior': 48,
 'letalidade_CISP22_atual': 150,
 'participacao_no_aumento_municipal_pct': np.float64(47.0),
 'mortes_intervencao_CISP22_out_2025': 118,
 'mortes_intervencao_CISP22_periodo': 125,
 'letalidade_Rio_sem_CISP22_variacao_pct': np.float64(8.7)}

## 3. Áreas de renda mais alta registram outro perfil de crime

Filtro: CISPs com ao menos 50 mil moradores. Comparam-se as sete maiores e as sete menores pelo proxy de renda. O proxy é a média ponderada das medianas setoriais do rendimento nominal mensal do responsável com rendimento (V06006); não é renda per capita nem a mediana oficial da CISP. Resultado exploratório, não causal.

In [4]:
context = population.join(income).join(territories[['territorialUnit']])
eligible = context.query('population >= 50000').sort_values('weightedSectorMedianIncomeProxy')
low = eligible.head(7).index
high = eligible.tail(7).index

metrics = ['total_furtos','furto_celular','estelionato','hom_doloso','roubo_veiculo','total_roubos']
metric_labels = {
    'total_furtos':'Furtos', 'furto_celular':'Furto de celular', 'estelionato':'Estelionato',
    'hom_doloso':'Homicídio doloso', 'roubo_veiculo':'Roubo de veículo', 'total_roubos':'Roubos'
}
def group_rates(indices):
    denominator = population.loc[indices, 'population'].sum()
    return current.loc[indices, metrics].sum() / denominator * 100000
comparison = pd.DataFrame({'renda_mais_alta': group_rates(high), 'renda_mais_baixa': group_rates(low)})
comparison['relacao_alta_sobre_baixa'] = comparison.renda_mais_alta / comparison.renda_mais_baixa
comparison.index = [metric_labels[i] for i in comparison.index]
print('Grupo de renda mais alta:', list(high))
print('Grupo de renda mais baixa:', list(low))
comparison.round(2)

Grupo de renda mais alta: [12, 13, 9, 10, 16, 15, 14]
Grupo de renda mais baixa: [11, 36, 43, 39, 21, 34, 31]


,renda_mais_alta,renda_mais_baixa,relacao_alta_sobre_baixa
Furtos,4657.17,625.07,7.45
Furto de celular,1535.14,110.59,13.88
Estelionato,2432.31,717.61,3.39
Homicídio doloso,6.56,21.40,0.31
Roubo de veículo,44.50,204.35,0.22
Roubos,847.92,647.81,1.31


## 4. Quantidade e taxa respondem a perguntas diferentes

Campo Grande lidera em carga total de registros, mas não em registros por residente. O Centro mostra o limite do denominador residencial em áreas de grande população flutuante.

In [5]:
rates = current[['registro_ocorrencias']].join(population[['population']]).join(territories[['territorialUnit']])
rates['taxa'] = rates.registro_ocorrencias / rates.population * 100000
rates['rank_quantidade'] = rates.registro_ocorrencias.rank(method='min', ascending=False).astype(int)
rates['rank_taxa'] = rates.taxa.rank(method='min', ascending=False).astype(int)
rates.loc[[35,1], ['territorialUnit','registro_ocorrencias','population','taxa','rank_quantidade','rank_taxa']].round(1)

,territorialUnit,registro_ocorrencias,population,taxa,rank_quantidade,rank_taxa
cisp,,,,,,
35,"Campo Grande, Cosmos, Inhoaíba, Santíssimo e S...",30425,602626,5048.7,1,35
1,Centro (parte),8760,1381,634323.0,30,1


## 5. “Crime patrimonial” muda de composição conforme o território

In [6]:
composition = current[['total_furtos','total_roubos']].join(territories[['territorialUnit']])
composition['furtos_por_roubo'] = composition.total_furtos / composition.total_roubos
composition.loc[[12,14,39,29], ['territorialUnit','total_furtos','total_roubos','furtos_por_roubo']].round(2)

,territorialUnit,total_furtos,total_roubos,furtos_por_roubo
cisp,,,,
12,Copacabana (parte) e Leme,5673,454,12.50
14,Ipanema e Leblon,6559,729,9.00
39,"Acari, Barros Filho, Costa Barros, Parque Colú...",1502,3366,0.45
29,"Cavalcanti, Engenheiro Leal, Madureira, Turiaç...",3435,3564,0.96


## 6. Inversões locais que um total esconderia

In [7]:
checks = []
for cisp, metrics_for_cisp in {18:['total_roubos','total_furtos','furto_celular'], 24:['roubo_veiculo','furto_veiculos']}.items():
    for metric in metrics_for_cisp:
        before, after = previous.loc[cisp,metric], current.loc[cisp,metric]
        checks.append({'cisp':cisp,'regiao':territories.loc[cisp,'territorialUnit'],'indicador':metric,'anterior':int(before),'atual':int(after),'variacao_pct':(after/before-1)*100})
pd.DataFrame(checks).round(1)

,cisp,regiao,indicador,anterior,atual,variacao_pct
0,18,"Maracanã, Praça da Bandeira e Tijuca (parte)",total_roubos,2630,2185,-16.9
1,18,"Maracanã, Praça da Bandeira e Tijuca (parte)",total_furtos,4288,5004,16.7
2,18,"Maracanã, Praça da Bandeira e Tijuca (parte)",furto_celular,1813,2478,36.7
3,24,"Abolição, Água Santa (parte), Encantado, Engen...",roubo_veiculo,848,660,-22.2
4,24,"Abolição, Água Santa (parte), Encantado, Engen...",furto_veiculos,145,304,109.7


## Limites e regras editoriais

- Abril a julho de 2026 estão em fase 2 do ISP e podem receber erratas trimestrais.
- A geografia é CISP; nenhum valor deve ser atribuído a um bairro isolado dentro dela.
- Taxa por residente não controla população flutuante.
- Correlação com renda não prova causalidade; renda é Censo 2022 e crime é 2025–26.
- Publicar quantidades e denominadores junto das taxas.
- Não afirmar que a Operação Contenção “causou” todos os registros de outubro; a base agregada não identifica cada ocorrência.